In [ ]:
# Olist late delivery - TIME-ORDERED train / test, 80% train / 20% test
# Simple beginner code: no function definitions, straight top to bottom.
# Input: the Orders_PowerBI file you already have (96,470 delivered orders).
# Output: prediction.csv - the newest 20% of orders only, in the same
# 13-column format as before, so Power BI can join it to Orders_PowerBI
# on order_id.
# This is NOT a random split. Orders are sorted by purchase_date first.
# The first 80% (oldest orders) are train, the last 20% (newest orders)
# are test. That is the real future test: test months never appear in train.
# Run this in Google Colab (T4 is fine, this is CPU tabular work).
# In Colab, first upload olist_powerbi_orders.csv to the session,
# or change FILE_PATH below to where your file is.

# STEP 1 - File path - change this one line if your file is somewhere else
FILE_PATH = "olist_powerbi_orders.csv"

# STEP 2 - Imports
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, confusion_matrix

# STEP 3 - Load the Orders_PowerBI dataset directly from the file
orders = pd.read_csv(FILE_PATH, parse_dates=["purchase_date"])
print("Orders loaded. Rows:", len(orders))
print("Columns:", list(orders.columns))
print("Late orders:", int(orders["actual_late"].sum()), " Late rate:", round(orders["actual_late"].mean(), 4))

# STEP 4 - Sort by purchase_date, oldest first, newest last
# This step is what makes it a time split, not a random split.
# After sorting, row order is time order, so the cut below is a date cut.
orders = orders.sort_values("purchase_date").reset_index(drop=True)
print("\nOldest order in file:", orders["purchase_date"].min())
print("Newest order in file:", orders["purchase_date"].max())

# STEP 5 - Choose features that are known at purchase time
# order_id is only an identifier, so it is not a feature.
# purchase_date is used only for sorting, it is not a feature.
# purchase_month already covers time as a category.
# actual_late is the target, so it is never a feature.
# No delivered dates, no review scores, no shipping dates are used.
text_features = ["purchase_month", "main_category", "main_seller_state", "customer_state", "payment_type"]
number_features = ["total_price", "total_freight", "payment_value", "item_count", "seller_count", "estimated_days"]
features = text_features + number_features
target = "actual_late"
print("Text features:", text_features)
print("Number features:", number_features)

X = orders[features]
y = orders[target]

# STEP 6 - Time-ordered split: first 80% train, last 20% test
# No random_state, no shuffling, no stratify here.
# The cut is a position in time order, not a random pick.
cut = int(len(orders) * 0.80)
X_train = X.iloc[:cut]
X_test = X.iloc[cut:]
y_train = y.iloc[:cut]
y_test = y.iloc[cut:]
orders_train = orders.iloc[:cut]
orders_test = orders.iloc[cut:]
print("\nTrain rows (first 80%, oldest):", len(X_train), " Late:", int(y_train.sum()), " Rate:", round(y_train.mean(), 4))
print("Train dates:", orders_train["purchase_date"].min(), " to ", orders_train["purchase_date"].max())
print("Test rows (last 20%, newest):", len(X_test), " Late:", int(y_test.sum()), " Rate:", round(y_test.mean(), 4))
print("Test dates:", orders_test["purchase_date"].min(), " to ", orders_test["purchase_date"].max())
print("Test months:")
print(orders_test["purchase_month"].value_counts().sort_index())

# STEP 7 - Turn text columns into numbers (one-hot encoding)
# Test columns are reindexed to the train columns, so a category that
# appears only in train or only in test does not break the model.
X_train_encoded = pd.get_dummies(X_train, columns=text_features)
X_test_encoded = pd.get_dummies(X_test, columns=text_features)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)
print("\nNumber of columns after one-hot encoding:", len(X_train_encoded.columns))

# STEP 8 - Scale numbers for Logistic Regression only
# Tree models (Random Forest, HistGradientBoosting) do not need scaling.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

# STEP 9 - Train Model 1: Logistic Regression (baseline)
print("\n--- Training Logistic Regression ---")
log_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_model.fit(X_train_scaled, y_train)
log_test_prob = log_model.predict_proba(X_test_scaled)[:, 1]
print("Logistic Regression on newest 20% test:")
print("ROC-AUC:", roc_auc_score(y_test, log_test_prob))
print("PR-AUC:", average_precision_score(y_test, log_test_prob))

# STEP 10 - Train Model 2: Random Forest
print("\n--- Training Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced")
rf_model.fit(X_train_encoded, y_train)
rf_test_prob = rf_model.predict_proba(X_test_encoded)[:, 1]
print("Random Forest on newest 20% test:")
print("ROC-AUC:", roc_auc_score(y_test, rf_test_prob))
print("PR-AUC:", average_precision_score(y_test, rf_test_prob))

# STEP 11 - Train Model 3: HistGradientBoosting (best for tabular data)
# This is the model used for prediction.csv below.
print("\n--- Training HistGradientBoosting ---")
hgb_model = HistGradientBoostingClassifier(random_state=42, max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20)
hgb_model.fit(X_train_encoded, y_train)
hgb_test_prob = hgb_model.predict_proba(X_test_encoded)[:, 1]
print("HistGradientBoosting on newest 20% test:")
print("ROC-AUC:", roc_auc_score(y_test, hgb_test_prob))
print("PR-AUC:", average_precision_score(y_test, hgb_test_prob))
print("Baseline PR-AUC (test late rate):", round(y_test.mean(), 4))

# STEP 12 - Threshold 0.10, same as the earlier models
# Late orders are only about 5% in this newest test set, so at
# threshold 0.5 the model would flag almost nothing. At 0.10 it flags
# a shortlist to review. It is not a certainty: most flagged orders
# will still be on time.
threshold = 0.10
hgb_test_pred = (hgb_test_prob >= threshold).astype(int)
print("\n--- HistGradientBoosting at threshold 0.10, on newest 20% test ---")
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, hgb_test_pred))
print("Precision:", round(precision_score(y_test, hgb_test_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, hgb_test_pred, zero_division=0), 4))
print("F1:", round(f1_score(y_test, hgb_test_pred, zero_division=0), 4))
print("Flagged as late:", int(hgb_test_pred.sum()), " out of", len(hgb_test_pred))
print("Actually late, caught:", int(((y_test.values == 1) & (hgb_test_pred == 1)).sum()), " out of", int(y_test.sum()))

# STEP 13 - Save prediction.csv in the same format as before
# Same 13 columns, same order, one row per test order only.
# Join this file to Orders_PowerBI in Power BI on order_id.
# This file replaces the random-split prediction.csv, do not load both.
test_results = orders_test[["order_id", "purchase_month", "main_category", "main_seller_state", "customer_state", "payment_type", "total_price", "total_freight", "estimated_days", "item_count"]].copy()
test_results["actual_late"] = y_test.values
test_results["predicted_probability"] = hgb_test_prob
test_results["predicted_late"] = hgb_test_pred
test_results.to_csv("prediction.csv", index=False)
print("\nSaved predictions: prediction.csv  Rows:", len(test_results))
print("Columns:", list(test_results.columns))
print(test_results.head())


Orders loaded. Rows: 96470
Columns: ['order_id', 'purchase_month', 'purchase_date', 'main_category', 'main_seller_state', 'customer_state', 'payment_type', 'total_price', 'total_freight', 'payment_value', 'item_count', 'seller_count', 'estimated_days', 'actual_late']
Late orders: 7826  Late rate: 0.0811

Oldest order in file: 2016-09-15 12:16:38
Newest order in file: 2018-08-29 15:00:37
Text features: ['purchase_month', 'main_category', 'main_seller_state', 'customer_state', 'payment_type']
Number features: ['total_price', 'total_freight', 'payment_value', 'item_count', 'seller_count', 'estimated_days']

Train rows (first 80%, oldest): 77176  Late: 6805  Rate: 0.0882
Train dates: 2016-09-15 12:16:38  to  2018-05-26 18:16:57
Test rows (last 20%, newest): 19294  Late: 1021  Rate: 0.0529
Test dates: 2018-05-26 18:18:03  to  2018-08-29 15:00:37
Test months:
purchase_month
2018-05     691
2018-06    6096
2018-07    6156
2018-08    6351
Name: count, dtype: int64

Number of columns after one-

In [ ]:
from google.colab import files
files.download("prediction.csv")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>